# NOAA/NCEI HazEL API — Exploration Notebook (Fixed)

**Bugs fixed from v1:**
1. `fetch_ncei_earthquakes()` now paginates correctly — the API returns 25 records per page by default, so without pagination you only ever got the first page.
2. The merge function now uses `eqMagnitude` (the actual NCEI column name) instead of `magnitude`.
3. The `time` column is built before the merge runs.

**API base URL:** `https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/earthquakes`  
**No API key required.**

In [1]:
import requests
import pandas as pd
import numpy as np
import json

BASE_URL = "https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/earthquakes"

## 1. Check the pagination structure

The API response has these keys: `items`, `page`, `totalPages`, `itemsPerPage`, `totalItems`.
We need to loop through all pages to get the full dataset.

In [2]:
# Peek at one page to understand the pagination fields
r = requests.get(BASE_URL, params={"minYear": 2020, "maxYear": 2020})
r.raise_for_status()
data = r.json()

print("Response keys:     ", list(data.keys()))
print("Items on this page:", len(data["items"]))
print("Total items:       ", data["totalItems"])   
print("Total pages:       ", data["totalPages"])
print("Items per page:    ", data["itemsPerPage"])

Response keys:      ['items', 'page', 'totalPages', 'itemsPerPage', 'totalItems']
Items on this page: 33
Total items:        33
Total pages:        1
Items per page:     200


## 2.fetch function

In [3]:
def fetch_ncei_earthquakes(min_year: int, max_year: int, min_magnitude: float = 0) -> pd.DataFrame:
    params = {
        "minYear": min_year,
        "maxYear": max_year,
        "page": 1,
    }
    if min_magnitude > 0:
        params["minMagnitude"] = min_magnitude

    all_items = []
    page = 1

    while True:
        params["page"] = page
        r = requests.get(BASE_URL, params=params)
        r.raise_for_status()
        data = r.json()

        items = data.get("items", [])
        all_items.extend(items)

        total_pages = data.get("totalPages", 1)
        print(f"  Page {page}/{total_pages} — {len(all_items)} records so far", end="\r")

        if page >= total_pages:
            break
        page += 1

    print(f"\nDone. Fetched {len(all_items)} total records ({min_year}–{max_year})")
    return pd.DataFrame(all_items)

## 3. Fetch the full 2000–2024 dataset

In [4]:
df_ncei = fetch_ncei_earthquakes(min_year=2000, max_year=2024)

print(f"\nShape: {df_ncei.shape}")
print(f"Columns: {df_ncei.columns.tolist()}")

  Page 8/8 — 1410 records so far
Done. Fetched 1410 total records (2000–2024)

Shape: (1410, 48)
Columns: ['id', 'year', 'month', 'day', 'hour', 'minute', 'second', 'locationName', 'latitude', 'longitude', 'eqDepth', 'eqMagnitude', 'damageAmountOrder', 'eqMagMb', 'publish', 'damageAmountOrderTotal', 'housesDamagedTotal', 'housesDamagedAmountOrderTotal', 'country', 'regionCode', 'injuries', 'injuriesAmountOrder', 'housesDestroyed', 'housesDestroyedAmountOrder', 'housesDamaged', 'housesDamagedAmountOrder', 'eqMagMw', 'eqMagMs', 'injuriesTotal', 'injuriesAmountOrderTotal', 'housesDestroyedTotal', 'housesDestroyedAmountOrderTotal', 'deaths', 'deathsAmountOrder', 'damageMillionsDollars', 'eqMagMl', 'deathsTotal', 'deathsAmountOrderTotal', 'damageMillionsDollarsTotal', 'tsunamiEventId', 'intensity', 'volcanoEventId', 'area', 'missing', 'missingAmountOrder', 'missingTotal', 'missingAmountOrderTotal', 'eqMagUnk']


In [ ]:
print("Year range:", df_ncei["year"].min(), "–", df_ncei["year"].max())
print("Records per year:")
print(df_ncei["year"].value_counts().sort_index())

Year range: 2000 – 2024
Records per year:
year
2000    38
2001    26
2002    62
2003    73
2004    79
2005    60
2006    65
2007    67
2008    77
2009    62
2010    64
2011    62
2012    53
2013    56
2014    56
2015    48
2016    54
2017    66
2018    71
2019    64
2020    33
2021    42
2022    51
2023    46
2024    35
Name: count, dtype: int64


## 4. Build a proper datetime column

NCEI stores date as separate year/month/day columns. We combine them here
**before** any merge or filtering.

In [6]:
def build_datetime(row):
    try:
        year  = int(row["year"])
        month = int(row["month"]) if pd.notna(row.get("month")) else 1
        day   = int(row["day"])   if pd.notna(row.get("day"))   else 1
        return pd.Timestamp(year=year, month=month, day=day, tz="UTC")
    except Exception:
        return pd.NaT

df_ncei["time"] = df_ncei.apply(build_datetime, axis=1)

print("Null times:", df_ncei["time"].isna().sum())
df_ncei[["time", "latitude", "longitude", "eqMagnitude", "locationName"]].head()

Null times: 0


,time,latitude,longitude,eqMagnitude,locationName
0,2000-01-03 00:00:00+00:00,22.132,92.771,4.6,INDIA-BANGLADESH BORDER: MAHESHKHALI
1,2000-01-11 00:00:00+00:00,40.498,122.994,5.1,CHINA: LIAONING PROVINCE
2,2000-01-14 00:00:00+00:00,25.607,101.063,5.9,CHINA: YUNNAN PROVINCE: YAOAN COUNTY
3,2000-02-02 00:00:00+00:00,35.288,58.218,5.3,"IRAN: BARDASKAN, KASHMAR"
4,2000-02-07 00:00:00+00:00,-26.288,30.888,4.5,SOUTH AFRICA; SWAZILAND: MBABANE-MANZINI


## 5. Explore the impact columns

In [7]:
impact_cols = ["deaths", "injuries", "damageMillionsDollars", "housesDestroyed", "housesDamaged"]

for col in impact_cols:
    df_ncei[col] = pd.to_numeric(df_ncei[col], errors="coerce")

df_ncei[impact_cols].describe()

,deaths,injuries,damageMillionsDollars,housesDestroyed,housesDamaged
count,568.000000,759.000000,252.000000,3.880000e+02,4.390000e+02
mean,1133.098592,1639.156785,1808.007262,2.561721e+04,3.168339e+04
std,14387.959068,16578.609192,7011.318255,2.768046e+05,2.700835e+05
min,1.000000,1.000000,0.300000,1.000000e+00,1.000000e+00
25%,1.000000,6.000000,20.750000,2.075000e+01,1.000000e+02
50%,3.000000,26.000000,100.000000,2.185000e+02,6.710000e+02
75%,12.000000,132.500000,503.750000,1.792000e+03,4.250000e+03
max,316000.000000,374171.000000,86000.000000,5.360000e+06,5.360000e+06


In [8]:
# What fraction of records have each impact field populated?
df_ncei[impact_cols].notna().mean().sort_values(ascending=False).to_frame("pct_populated")

,pct_populated
injuries,0.538298
deaths,0.402837
housesDamaged,0.311348
housesDestroyed,0.275177
damageMillionsDollars,0.178723


In [9]:
# Deadliest earthquakes 2000–2024
df_ncei.nlargest(10, "deaths")[["year", "locationName", "eqMagnitude", "eqDepth", "deaths", "damageMillionsDollars"]]

,year,locationName,eqMagnitude,eqDepth,deaths,damageMillionsDollars
585,2010,HAITI: PORT-AU-PRINCE,7.0,13.0,316000.0,8000.0
465,2008,CHINA: SICHUAN PROVINCE,7.9,10.0,87652.0,86000.0
306,2005,"PAKISTAN: MUZAFFARABAD, URI, ANANTNAG, BARAMULA",7.6,15.0,76213.0,6680.0
1294,2023,TURKEY: KAHRAMANMARAS; SYRIA,7.8,17.0,56697.0,42900.0
182,2003,"IRAN: SOUTHEASTERN: BAM, BARAVAT",6.6,10.0,31000.0,500.0
36,2001,"INDIA: GUJARAT: BHUJ, AHMADABAD, RAJOKOT; PA...",7.6,17.0,20005.0,2623.0
867,2015,NEPAL: KATHMANDU; INDIA; CHINA; BANGLADESH,7.8,8.0,8957.0,6000.0
343,2006,"INDONESIA: JAVA: BANTUL, YOGYAKARTA",6.3,13.0,6234.0,3100.0
1071,2018,INDONESIA: SULAWESI,7.5,20.0,4340.0,1500.0
604,2010,CHINA: QINGHAI PROVINCE: YUSHU,6.9,17.0,2968.0,500.0


## 6. Save the full NCEI dataset

In [ ]:
df_ncei.to_csv("ncei_earthquakes_2000_2024.csv", index=False)

Saved 1410 records to ncei_earthquakes_2000_2024.csv


## 7. Prototype the USGS ↔ NCEI merge

Strategy: for each NCEI significant earthquake, find the closest USGS event by:
1. Time window: ±3 days
2. Spatial proximity: within ~1 degree lat/lon (≈111 km)